# 01 · The intraday close regime — mechanism & how it was found

*Edge-features arc · 01 discovery · 02 linear base · 03 hero cascade · 04 regime-MoE · 05 temporal-kNN  —  machinery: `ridge_pipeline_throughline.ipynb`*

**Verify first / interpret second.** This chapter is the *why*: what the residualized-EBM campaign
actually learned, and the falsifiable loop that surfaced it. Every number is a cluster walk-forward on
the slim `all_buckets` cell (full-OOS Duan-smeared QLIKE); the discovery scripts are named for reproduction.

The headline: the residual the tuned tree extracts is an **intraday auction/session-transition regime** —
HAR volatility-persistence **sign-flips at the session edges** — not a leftover diurnal U-shape.

In [ ]:
import html, inspect, json, os, sys, textwrap
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

def find_repo(s):
    for q in [Path(s).resolve(), *Path(s).resolve().parents]:
        if (q / "resid_amortized.py").exists() and (q / "src").is_dir():
            return q
    raise FileNotFoundError("repo root")
REPO = find_repo(Path.cwd()); os.chdir(REPO); sys.path.insert(0, str(REPO))

def _details(f, open_=False):
    mod = f.__module__.replace("src.", "src/").replace(".", "/") + ".py"
    try:
        sig = ("class " + f.__name__) if inspect.isclass(f) else ("def " + f.__name__ + str(inspect.signature(f)))
    except (ValueError, TypeError):
        sig = f.__qualname__
    body = "```python\n" + textwrap.dedent(inspect.getsource(f)).rstrip() + "\n```"
    return (f"<details{' open' if open_ else ''}>\n<summary><code>{html.escape(mod + '  ·  ' + sig)}"
            f"</code></summary>\n\n{body}\n\n</details>")
def show_one(f):
    return Markdown(_details(f))
print("setup ok")

---
## 1 · The discovery loop (the asset)

The close regime was found by a **fixed 6-step loop**, run once on the `hour` axis. Reconstructed
(`writeup/discovery_process_methodology_2026-06-29.md`):

| # | step | tool / tell | output |
|---|---|---|---|
| 1 | black box beats the linear base | EBM on residual **0.12414** vs enet **0.12516** | the residual has structure |
| 2 | read the black box | `ebm_interpret.py` | `hour` jumps enet \|coef\| rank **105 → EBM rank 7**; 3/5 top interactions involve `hour` |
| 3 | test the interaction OOS | `regime_study.py` / `har_flip.py` | clock-regime **+0.0095 OOS R²**; vol-state **−0.0027 (fails)**; HAR persistence flips 5/6 windows |
| 4 | localize sharply | `tests123.py` — `corr(har_ma_5, resid)` by hour | **−0.085 at h9 (open)**, **−0.21 at h17 (close/AH)**; sharp, not gradual |
| 5 | distill to one feature | `distill_regime.py` | `HAR × late-day` recovers **58%** of the tree's edge |
| 6 | falsification gauntlet | U-control, gamma, real-space, FORCE-past-L1 | confounds ruled out; ~38% genuinely higher-order |

## 2 · The compass — the tree-subsumption law

A boosted tree / EBM is **invariant to monotone transforms of the current row**. So the *only* features
that beat a fitted tree are **functionals of history/sequence the row doesn't contain**. This tells you
where to look (history-dependent regime functionals) and when you're done (when the only surviving lever
is 5th-decimal → the lever is *data*, not features). It also frames everything downstream: the linear
improvers (ch. 02) get **absorbed** by the tree; the regime stage (ch. 03) is what survives.

## 3 · Interpret — a session-edge vol regime

- **The close DAMPS short-horizon persistence** — read off the legible unpenalized-HAR base:
  `har_ma_5×close` coef = **−0.05** (high recent vol → close forecast pulled *below* HAR's extrapolation).
- **The intraday vol PATH matters at the close** — `cumrv×close` is the single biggest engineered edge
  (base **0.12436 → 0.12314, −0.00122**); mechanistically the **sqrt vol-scale accumulation** (97.5% of it).
- **The open is a discrete auction/gap event** — a symmetric overnight-cumrv-at-open feature is null.
- **Clock-anchored, not vol-state-anchored**; dealer-gamma is a weak (~20%) modulator.

⇒ the residual edge is the equilibrium footprint of auction / MOC liquidity + dealer hedging — tiny,
already-arbitraged, strongest where costs are highest. The deliverable is the **mechanism + a map of
which data to buy** (auction imbalance / GEX / OFI), not the 4th-decimal QLIKE. Continues in **ch. 02**.